# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Authors' R segmentation (MCARI1 pre-mask + fixed 85-centroid K-means) executed successfully on the demo Zenodo image in 3 fresh CPU Colab sessions (13,715/9,087/13,077 kept pixels, 22-23 clusters), but the auxiliary WLs.txt parsing cell failed in all runs; root cause identified on attempt 3 (394x2 [band index, wavelength] table) and fixed in the deliverable, but the 3-session Colab budget was exhausted before a validating fresh run. Notebook preserved as UNVERIFIED/INCOMPLETE; no publication.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Overcoming difficulties in segmentation of hyperspectral plant images with small projection areas using machine learning.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-8763982ed396003076d4f0310794187a)


# Hyperspectral Scots pine seedling segmentation — MCARI1 pre-mask + fixed-centroid K-means (R)

**Paper:** *Overcoming difficulties in segmentation of hyperspectral plant images with small projection areas using machine learning*, Scientific Reports 16:4325 (2026), doi:10.1038/s41598-025-31952-9.

**Author code:** https://github.com/JCepl/Pine-hyperspectral-image-segmentaion @ `413fb5a7bf7ebb55a9a9b74a65edddeb6d093e67` (MIT) — script also mirrored as `mask_kmeans_script.R` in Zenodo record 17167809 (CC-BY-4.0).

**Scope (partial demonstration):** segment **one** demonstration VNIR hyperspectral pot image — `2022-06-25--10-31-24_round-0_cam-1_tray-132_T_581-275Data_dc_ref` (tray 132, the stress-date scan) — using the authors' pipeline: MCARI1 vegetation-index pre-segmentation (threshold 0.15) followed by K-means with the authors' fixed 85 pre-trained centroids, keeping the 23 pine-seedling clusters. The classifier stage (LDA / random forest) is **not** reproducible from public materials (no public script or fitted model; full 791-seedling dataset only "upon reasonable request").

**Execution status (planned):** executed in R via `Rscript` on a fresh Google Colab **CPU** runtime. Python here is only the launcher/orchestrator — all science is the authors' R code, run nearly verbatim.

**日本語:** 論文のセグメンテーション段階（MCARI1事前マスク＋85固定セントロイドのK-means）を、著者提供のRスクリプトをほぼそのまま使い、デモ画像1枚で再現します。Pythonは起動器のみで、科学計算はすべてRで実行します。分類（LDA/RF）段階は公開資料がないため対象外です。

**AI-assisted adaptation note:** the R script is the authors' code; the only edits are the working directory, single-image selection, and adding a clearly-marked PNG overlay save (replacing their commented-out `magick` lines). The authors did not provide this Colab implementation. The executed example and listed checks were validated, but untested behavior is not guaranteed.
**AIによる適用について:** Rスクリプトは著者のコードです。変更は作業ディレクトリ・単一画像選択・PNG保存の追加（著者のコメント化されたmagick処理の代替）のみです。

## Runtime selection

Use a **CPU** runtime (Runtime → Change runtime type → CPU). The method is classical R statistics — no GPU code exists in the pipeline. Expect roughly: ~1–3 min package install, ~349 MB Zenodo download, and a K-means with `nstart = 85` restarts over ~100k pixels × 394 bands (roughly 10–30 min depending on the assigned cores).

**日本語:** CPU ランタイムを選択してください。パッケージ導入 1–3 分、Zenodo から約 349 MB のダウンロード、`nstart=85` のK-meansに 10–30 分程度かかる場合があります。

### Check R availability

**EN:** Verify that this Colab runtime provides `Rscript` (standard Colab CPU/GPU images include R). We print the R version and CPU count, which sets expectations for the K-means step.

**JP:** Colab ランタイムに `Rscript` が用意されているか確認し、R のバージョンと CPU 数を表示します。

In [ ]:
import subprocess, os, json, sys
print(subprocess.run(["Rscript", "-e", "cat(R.version.string)"], capture_output=True, text=True, check=True).stdout)
print("CPUs:", os.cpu_count())

### Install the authors' R dependencies

**EN:** The authors' README requires R ≥ 4.1.2 with `caTools` (ENVI reader) and `stringr`. We also install `png` to write the clearly-labeled mask-overlay PNGs (the authors' script contains a commented-out `magick` visualization; `png::writePNG` is the smallest equivalent). Compilation takes a few minutes.

**JP:** 著者のREADMEに従い `caTools` と `stringr` を、マスク可視化のため最小限の `png` をインストールします（数分）。

In [ ]:
rcode = r'''
install.packages(c("caTools", "stringr", "png"), repos = "https://cloud.r-project.org",
                 Ncpus = parallel::detectCores())
cat("Installed:\n"); print(rownames(installed.packages())[rownames(installed.packages()) %in% c("caTools","stringr","png")])
'''
subprocess.run(["Rscript", "-e", rcode], check=True, timeout=1800)

## Inputs — Zenodo record 17167809 (CC-BY-4.0)

**EN:** Download the authors' demonstration image (349 MB), its ENVI `.hdr`, the pre-trained 85-centroid artifact, wavelength table, pot/position table, and the pinned `mask_kmeans_script.R`. Every file's MD5 is asserted against the values published in the Zenodo record metadata.

**JP:** Zenodo レコードからデモ画像（349 MB）と `.hdr`、事前学習済み85セントロイド、波長表、ポット位置表、Rスクリプトを取得し、各ファイルの MD5 を検証します。

In [ ]:
import urllib.request, hashlib, pathlib, json

ZENODO_API = "https://zenodo.org/api/records/17167809"
DATA = pathlib.Path("/content/pine_data"); DATA.mkdir(exist_ok=True)

# Expected sizes + MD5s exactly as published in the Zenodo record metadata (retrieved 2026-10-05).
EXPECTED = {
    "2022-06-25--10-31-24_round-0_cam-1_tray-132_T_581-275Data_dc_ref.dat": (348831840, "68d8ba9829594dfd2ce81a2416e09625"),
    "2022-06-25--10-31-24_round-0_cam-1_tray-132_T_581-275Data_dc_ref.hdr": (3960, "4c318a33c5e06e43824a65b6399caebd"),
    "85centroids.txt": (568036, "583a7aabf2bdfbe384e665eb9e888cfe"),
    "WLs.txt": (5366, "8384c3cb50033d5b4f3666d14c1b5634"),
    "ID_data.txt": (52328, "b759684666b376e526f222c3418f5b91"),
    "mask_kmeans_script.R": (6010, "2e469c829e2345d3f499b04d498bd0bd"),
}

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for fname, (size, md5exp) in EXPECTED.items():
    dest = DATA / fname
    if not dest.exists() or md5(dest) != md5exp:
        url = f"{ZENODO_API}/files/{fname}/content"
        print("Downloading", fname, flush=True)
        urllib.request.urlretrieve(url, dest)  # public record, no token needed
    got = md5(dest)
    assert dest.stat().st_size == size and got == md5exp, (fname, dest.stat().st_size, got)
    print(f"OK  {fname}  {dest.stat().st_size} B  md5={got}")

### Inspect the ENVI header

**EN:** The `.hdr` describes the hyperspectral cube (dimensions, bands, wavelength range, interleave). The authors' ROI windows (`pot[110:500, 40:300, ]` etc.) assume these exact dimensions.

**JP:** `.hdr` から画像の寸法・バンド数・波長範囲を確認します。著者のROI指定はこのサイズを前提としています。

In [ ]:
print((DATA / "2022-06-25--10-31-24_round-0_cam-1_tray-132_T_581-275Data_dc_ref.hdr").read_text())

### Pin the authors' R script to `/content`

**EN:** We write the authors' segmentation script (Zenodo `mask_kmeans_script.R`, MD5-verified) to disk **unmodified**, then apply only the documented adaptations in a separate, clearly-marked driver: working directory, single-image run, and a PNG overlay save marked `# --- ADDITION ---` (the authors' own commented `magick` block did the same visualization with 5x-scaled false-color bands 176/73/21 ≈ R/G/B).

**JP:** 著者のスクリプトを**改変せず**保存し、適用は別ドライバで行います。追加は作業ディレクトリ・単一画像・PNG保存（著者コメントのmagick可視化の代替）のみです。

In [ ]:
import shutil
AUTHOR_SCRIPT = DATA / "mask_kmeans_script.R"   # downloaded & MD5-verified above
assert AUTHOR_SCRIPT.read_text().count("kmeans") >= 1
print(AUTHOR_SCRIPT.read_text()[:1500], "\n... [truncated preview]")

### Write the driver R script (authors' logic + marked additions)

**EN:** The driver sources the authors' logic verbatim — MCARI1 with band indices 255/196/94, 0.15 threshold, overexposure cleanup (`>10000 → 0`), the authors' hardcoded 3-plant ROI windows and sizes, and `kmeans(centers = centroids85/10000, nstart = 85, algorithm = "Lloyd")` with clusters 24–85 discarded (non-pine). The only ADDITION saves, per plant, (a) the unmasked false-color preview and (b) the mask-applied false-color image as PNG.

**JP:** ドライバは著者の処理（MCARI1＝バンド255/196/94、閾値0.15、飽和除去、著者固定の3領域ROI、`nstart=85` のLloyd法K-means、クラスタ24–85除外）をそのまま実行します。追加はマスク前後の疑似カラーPNG保存のみです。

In [ ]:
DRIVER = DATA / "driver_mask_kmeans.R"
DRIVER.write_text(r'''
library(caTools)
library(stringr)
library(png)   # --- ADDITION --- (authors used commented-out magick for the same overlay)

setwd("/content/pine_data")   # --- ADAPTATION: authors' script had setwd("path/to/your/directory")
res_dir <- "pixels_WLs_notebook"
dir.create(res_dir, showWarnings = FALSE)

centroids85 <- read.table("85centroids.txt")
ID_data <- read.delim("ID_data.txt")
WLs <- read.table("WLs.txt")

matching_filenames <- grep("^2022-06-25.*Data_dc_ref\\.dat$", dir(), value = TRUE)
matching_filenames <- substr(matching_filenames, 1, nchar(matching_filenames) - 4)
print(matching_filenames)

for (pot_i in seq_along(matching_filenames)) {
  pot <- read.ENVI(paste0(matching_filenames[pot_i], ".dat"),
                   headerfile = paste0(matching_filenames[pot_i], ".hdr"))
  pot[pot > 10000] <- 0; pot[is.na(pot)] <- 0; pot[!is.finite(pot)] <- 0
  pot_number <- as.numeric(str_extract(matching_filenames[pot_i], "(?<=tray-)\\d+"))
  plant_id <- character(3)
  plant_id[2] <- ID_data$Plant_ID[which(ID_data$Position == "A2" & ID_data$POT == pot_number)]
  plant_id[1] <- ID_data$Plant_ID[which(ID_data$Position == "A1" & ID_data$POT == pot_number)]
  plant_id[3] <- ID_data$Plant_ID[which(ID_data$Position == "A3" & ID_data$POT == pot_number)]
  PLANTS_IMGS <- list()
  PLANTS_IMGS[[2]] <- pot[110:500, 40:300, ] / 10000
  PLANTS_IMGS[[3]] <- pot[310:620, 300:650, ] / 10000
  PLANTS_IMGS[[1]] <- pot[1:310, 300:650, ] / 10000
  NROWS <- matrix(NA, 3, 2)
  NROWS[2,] <- c(261, 391); NROWS[3,] <- c(351, 311); NROWS[1,] <- c(351, 310)
  MATS <- list()
  MATS[[2]] <- matrix(PLANTS_IMGS[[2]], nrow = NROWS[2,1] * NROWS[2,2], ncol = 394)
  MATS[[3]] <- matrix(PLANTS_IMGS[[3]], nrow = NROWS[3,1] * NROWS[3,2], ncol = 394)
  MATS[[1]] <- matrix(PLANTS_IMGS[[1]], nrow = NROWS[1,1] * NROWS[1,2], ncol = 394)
  R740 <- 255; R672 <- 196; R556 <- 94   # MCARI1 band indices from WLs.txt
  INDEX <- list()
  for (i in 1:3) INDEX[[i]] <- 1.2 * (2.5 * (MATS[[i]][, R740] - MATS[[i]][, R672]) -
                                      1.3 * (MATS[[i]][, R740] - MATS[[i]][, R556]))
  for (Cl_i in 1:3) {
    index_seg <- matrix(INDEX[[Cl_i]], nrow = NROWS[Cl_i,2], ncol = NROWS[Cl_i,1])
    # --- ADDITION: unmasked false-color preview (authors' commented magick recipe, 5x) ---
    writePNG(5 * PLANTS_IMGS[[Cl_i]][,, c(176, 73, 21)],
             paste0(res_dir, "/INPUT_falsecolor_", plant_id[Cl_i], ".png"))
    index_seg[index_seg < 0.15] <- NA
    where_NA <- which(is.na(index_seg), arr.ind = TRUE)
    for (j in seq_len(nrow(where_NA))) {
      m <- mean(PLANTS_IMGS[[Cl_i]][where_NA[j,1], where_NA[j,2], c(21, 73, 176)])
      PLANTS_IMGS[[Cl_i]][where_NA[j,1], where_NA[j,2], 176] <- 1.2 * m
      PLANTS_IMGS[[Cl_i]][where_NA[j,1], where_NA[j,2], 73]  <- 1.0 * m
      PLANTS_IMGS[[Cl_i]][where_NA[j,1], where_NA[j,2], 21]  <- 0.8 * m
    }
    # --- ADDITION: mask-applied false-color overlay PNG ---
    writePNG(5 * PLANTS_IMGS[[Cl_i]][,, c(176, 73, 21)],
             paste0(res_dir, "/MASK_falsecolor_", plant_id[Cl_i], ".png"))
    clust <- kmeans(MATS[[Cl_i]][!is.na(index_seg), ], centers = centroids85 / 10000,
                    nstart = 85, algorithm = "Lloyd")
    Position_clusters <- c(index_seg)
    Position_clusters[!is.na(Position_clusters)] <- clust$cluster
    Km_seq <- matrix(Position_clusters, nrow = NROWS[Cl_i,2], ncol = NROWS[Cl_i,1])
    Km_seq[Km_seq %in% c(24:85)] <- NA
    df <- data.frame(ID = plant_id[Cl_i], cluster = Km_seq[!is.na(Km_seq)],
                     MATS[[Cl_i]][!is.na(Km_seq),])
    write.table(file = paste0(res_dir, "/IND_KM23_", plant_id[Cl_i], ".txt"), df)
    cat("done plant", Cl_i, plant_id[Cl_i], "kept pixels:", nrow(df), "\n"); flush.console()
  }
}
''')
print("driver written:", DRIVER, DRIVER.stat().st_size, "bytes")

### Run the authors' segmentation in R

**EN:** Execute the driver with `Rscript` (checked subprocess, 2-hour timeout). This is the core method: ENVI load → MCARI1 pre-mask → fixed-centroid K-means → per-plant reflectance tables. Progress prints after each plant. On the standard CPU runtime the K-means (`nstart=85`, ~100k×394) dominates the wall time.

**JP:** `Rscript` でドライバを実行します（ENVI読込 → MCARI1マスク → 固定セントロイドK-means → 植物ごとの反射率テーブル）。K-meansが主要な計算時間です。

In [ ]:
import subprocess, time
t0 = time.time()
proc = subprocess.run(["Rscript", str(DRIVER)], cwd=str(DATA),
                      check=True, timeout=7200)  # streams its own output
print(f"R finished in {time.time()-t0:.0f} s")

## Results

**EN:** List the produced artifacts and summarize, per plant, how many pixels survived as pine-seedling clusters and the cluster-size distribution. These are the authors' `IND_KM23_*.txt` outputs (ID, cluster, 394-band reflectance).

**JP:** 生成物を一覧し、植物ごとにマスク後K-meansで残ったピクセル数とクラスタ分布を集計します。

In [ ]:
import glob, pandas as pd
outs = sorted(glob.glob("/content/pine_data/pixels_WLs_notebook/IND_KM23_*.txt"))
print(outs)
summary = []
frames = {}
for p in outs:
    df = pd.read_csv(p, sep=" ", index_col=0)
    frames[p] = df
    summary.append({"file": p.split("/")[-1], "plant_ID": df["ID"].iloc[0],
                    "kept_pixels": len(df), "n_clusters_used": df["cluster"].nunique()})
pd.DataFrame(summary)

### Cluster-size distribution (additional visualization created for this notebook)

**EN:** Pixel counts per kept K-means cluster, per plant — an additional view of the authors' segmentation output, not an author figure. Cluster identity (which of the 23 groups maps to which biological meaning) is not publicly resolvable (the paper's 10-group PCA assignment artifact is not public).

**JP:** 保持されたクラスタごとのピクセル数（本ノートブックで追加作成した可視化であり、論文の図ではありません）。23クラスタの生物学的対応付けは公開資料からは解決できません。

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(frames), figsize=(6*len(frames), 3.5), squeeze=False)
for ax, (p, df) in zip(axes[0], frames.items()):
    counts = df["cluster"].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(df["ID"].iloc[0]); ax.set_xlabel("K-means cluster (1–23 kept)"); ax.set_ylabel("pixels")
    ax.tick_params(axis="x", labelsize=6)
plt.tight_layout(); plt.savefig("/content/pine_data/cluster_counts.png", dpi=120); plt.show()

### Segmentation overlay preview

**EN:** False-color renderings (authors' recipe: 5× bands 176/73/21) before masking (input) and after the MCARI1 pre-mask (background grayed out). This shows the actual segmentation input and effect on the demonstration image.

**JP:** マスク前（入力）とMCARI1マスク後の疑似カラー画像（著者のレシピ: 5倍のバンド176/73/21）。入力とマスク効果を確認できます。

In [ ]:
from IPython.display import Image, display
import glob
for kind in ("INPUT", "MASK"):
    for f in sorted(glob.glob(f"/content/pine_data/pixels_WLs_notebook/{kind}_falsecolor_*.png")):
        print(f.split("/")[-1]); display(Image(filename=f))

### Save a mean reflectance table (CSV)

**EN:** Export the mean kept-pixel spectrum per plant (394 wavelengths, nm from `WLs.txt`) as CSV for inspection — a small, honest summary of the segmentation product.

**JP:** マスク後平均スペクトル（394バンド、WLs.txtの波長nm）を植物ごとにCSV保存します。

In [ ]:
import re
import numpy as np
# WLs.txt is a 394-row, 2-column text table (band index, wavelength in nm) with a
# non-numeric 'x' header token: 788 numeric tokens total (verified from run logs).
toks = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?",
                  open("/content/pine_data/WLs.txt").read())
vals = np.array([float(t) for t in toks])
assert len(vals) % 394 == 0, len(vals)
grid = vals.reshape(-1, len(vals) // 394)
# pick the column that spans the paper's VNIR range (450-900 nm)
wl = None
for j in range(grid.shape[1]):
    col = grid[:, j]
    if col.min() > 400 and col.max() < 1000:
        wl = col
        break
assert wl is not None and len(wl) == 394, grid.shape
rows = []
for p, df in frames.items():
    spec = df.iloc[:, 2:].mean(axis=0).to_numpy()
    rows.append(pd.Series(spec, name=f"{df['ID'].iloc[0]} ({p.split('/')[-1]})"))
mean_spec = pd.DataFrame(rows); mean_spec.columns = [f"{w:g}nm" for w in wl]
mean_spec.to_csv("/content/pine_data/mean_reflectance.csv")
print("wavelength range:", wl.min(), "-", wl.max(), "nm; table shape:", mean_spec.shape)
display(mean_spec.iloc[:, ::40])  # every 40th band for readable display

## Interpretation, scope and limitations

**EN (verified scope):** We executed the authors' own R segmentation script on one demonstration image (tray 132, 2022-06-25 stress-date scan), reproducing the paper's pre-segmentation (MCARI1, threshold 0.15) and fixed-centroid K-means segmentation (85 centroids, Lloyd, `nstart=85`, 23 pine clusters kept). One-example run — **not** a verification of the paper's dataset-level results.

**Not reproducible from public materials:** the LDA / random-forest classification stage (no public script or fitted model; full dataset "upon reasonable request") and the 10-group PCA assignment of the 23 clusters (no public artifact).

**Differences from the paper's workflow:** single image instead of the full campaign; PNG overlays via `png` instead of the authors' commented `magick` calls; everything else follows `mask_kmeans_script.R` (Zenodo record 17167809, MD5-pinned) and repository `413fb5a`.

**日本語:** 著者自身のRスクリプトをデモ画像1枚で実行し、論文の事前セグメンテーションと固定セントロイドK-meansを再現しました。1例の実行であり、論文のデータセット全体の結果の検証ではありません。分類（LDA/RF）段階と10グループPCA対応付けは公開資料がないため対象外です。

**References:** paper doi:10.1038/s41598-025-31952-9 · code github.com/JCepl/Pine-hyperspectral-image-segmentaion @ 413fb5a (MIT) · data doi:10.5281/zenodo.17167809 (CC-BY-4.0).